# Notebook 5: Reconciliation and Evaluation

## Theoretical Foundation
This notebook implements the final, most critical components of modern forecasting theory:
1. **Hierarchical Reconciliation (Section 2.10.1):** We mathematically adjust the independent base forecasts so that the lower-level forecasts sum up exactly to the upper-level forecasts. We use a simple Bottom-Up approach to demonstrate the concept.
2. **Evaluation (Section 2.12.6):** We use the Diebold-Mariano test to prove our model is statistically significantly better than a baseline.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

preds_df = pd.read_parquet('data/04_ensemble_probabilistic_preds.parquet')

### 1. Back-Transformation
We must revert the log1p transformation to evaluate on the original scale.

In [ ]:
def invert_log1p(series):
    # log1p back-transform is expm1
    # We use np.clip to avoid infinite exponentials just in case
    return np.expm1(np.clip(series, a_min=0, a_max=20))

for col in ['demand_transformed', 'pred_mean', 'pred_q10', 'pred_q90']:
    new_col = col.replace('_transformed', '') if 'transformed' in col else col + '_orig'
    preds_df[new_col] = invert_log1p(preds_df[col])

# Rename for clarity
preds_df.rename(columns={'demand': 'actuals'}, inplace=True)


### 2. Hierarchical Reconciliation (Bottom-Up)
To ensure structural consistency ($Total = \sum Categories$), we reconcile the forecasts. Here we implement Bottom-Up: we take the base level (Category) forecasts and sum them to obtain the reconciled Total.

In [ ]:
# Separate base level (Category) and aggregate level (Total)
base_preds = preds_df[preds_df['series_id'].str.startswith('CAT_')].copy()

# Bottom-Up Reconciliation
reconciled_total = base_preds.groupby('month')[['pred_mean_orig', 'pred_q10_orig', 'pred_q90_orig']].sum().reset_index()
reconciled_total['series_id'] = 'TOTAL_RECONCILED'

# Compare original Total vs Reconciled Total
original_total = preds_df[preds_df['series_id'] == 'TOTAL'].sort_values('month')

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(original_total['month'], original_total['actuals'], 'k-', label='Actual Total', linewidth=2)
ax.plot(original_total['month'], original_total['pred_mean_orig'], 'r--', label='Base Total Forecast')
ax.plot(reconciled_total['month'], reconciled_total['pred_mean_orig'], 'b-.', label='Reconciled Total (Bottom-Up)')
ax.fill_between(reconciled_total['month'], reconciled_total['pred_q10_orig'], reconciled_total['pred_q90_orig'], color='blue', alpha=0.1, label='80% PI (Reconciled)')
ax.set_title('Hierarchical Reconciliation: Base vs Reconciled Total')
ax.legend()
plt.savefig('plots/04_reconciliation.png')
plt.show()

### 3. Statistical Testing: Diebold-Mariano
*Reference: Section 2.12.6 - Statistical tests of forecast performance.*
We evaluate if the ensemble model significantly outperforms a Seasonal Naive baseline.

In [ ]:
# Function to compute basic DM statistic for squared errors
def dm_test(actual, pred1, pred2):
    e1 = actual - pred1
    e2 = actual - pred2
    d = e1**2 - e2**2
    d_mean = np.mean(d)
    d_var = np.var(d, ddof=1)
    stat = d_mean / np.sqrt(d_var / len(d))
    return stat

# For demonstration, we use lag_12 (Seasonal Naive) as baseline
# It needs to be back-transformed too
preds_df['naive_orig'] = invert_log1p(preds_df['lag_12'])

# Calculate WAPE
def wape(actual, pred):
    return np.sum(np.abs(actual - pred)) / max(np.sum(actual), 1e-9)

print(f"Ensemble WAPE: {wape(preds_df['actuals'], preds_df['pred_mean_orig']):.4f}")
print(f"Naive WAPE: {wape(preds_df['actuals'], preds_df['naive_orig']):.4f}")

# Diebold-Mariano test on the Total series
total_mask = preds_df['series_id'] == 'TOTAL'
actual_tot = preds_df.loc[total_mask, 'actuals'].values
pred_tot = preds_df.loc[total_mask, 'pred_mean_orig'].values
naive_tot = preds_df.loc[total_mask, 'naive_orig'].values

dm_stat = dm_test(actual_tot, pred_tot, naive_tot)
print(f"\nDiebold-Mariano Statistic (Total Series): {dm_stat:.3f}")
if dm_stat < -1.96:
    print("Conclusion: The Ensemble is STATISTICALLY SIGNIFICANTLY better than Seasonal Naive (p < 0.05).")
else:
    print("Conclusion: No significant difference.")
